In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

# SCD Type 2 — Step by Step

**When:** Bronze → Silver, after transformation and deduplication.
Bronze stays append-only; the SCD2 logic lives entirely in the Silver load.

Columns condensed to the ones that change. `outlet_type` changes for `HCCB10002`,
`city` changes for `SLMG10003`. `customer_name`, `outlet_segment`, `state` etc. are
carried along unchanged.

Timestamps abbreviated: **T1 = 10:04:39**, **T2 = 11:10:21**.

---

## The pipeline

| # | Step | One-line code |
|---|---|---|
| 1 | Read Bronze | `bronze_df = spark.table("...bronze.scd2")` |
| 2 | Filter to the new window | `.filter(F.col("read_timestamp") > watermark_ts)` |
| 3 | Deduplicate to latest per key | `.withColumn("rn", F.row_number().over(w)).filter("rn = 1")` |
| 4 | Add SCD2 columns | `.withColumn("start_date", F.col("read_timestamp")).withColumn("end_date", F.lit(None)).withColumn("is_current", F.lit(True))` |
| 5 | **Close changed rows** | `.whenMatchedUpdate(condition=<any column differs>, set={"end_date": "source.start_date", "is_current": "false"})` |
| 6 | **Insert new versions** | `.whenNotMatchedInsertAll()` |

Steps 5 and 6 are **two separate merges**. One MERGE cannot both close a row and insert
its replacement — that is the whole reason SCD2 is a two-step operation while SCD1 is one.

---

## Starting point

### Bronze (append-only — both files, 8 rows)

| customer_code | outlet_type | city | read_timestamp |
|---|---|---|---|
| HCCB10001 | Bronze | Lucknow | T1 |
| HCCB10002 | Gold | Pune | T1 |
| SLMG10003 | Iron | Jaipur | T1 |
| SLMG10004 | Platinum | Gurugram | T1 |
| HCCB10005 | Bronze | Indore | T1 |
| 🟡 HCCB10002 | 🟡 **Platinum** | Pune | 🟡 **T2** |
| 🟡 SLMG10003 | Iron | 🟡 **Jodhpur** | 🟡 **T2** |
| 🟢 HCCB10006 | Gold | Ahmedabad | 🟢 **T2** |

`HCCB10002` and `SLMG10003` appear **twice** — once per file. That is correct.
Bronze records what arrived; it never updates.

### Silver before the incremental run (5 rows, all open)

| customer_code | outlet_type | city | start_date | end_date | is_current |
|---|---|---|---|---|---|
| HCCB10001 | Bronze | Lucknow | T1 | null | TRUE |
| HCCB10002 | Gold | Pune | T1 | null | TRUE |
| SLMG10003 | Iron | Jaipur | T1 | null | TRUE |
| SLMG10004 | Platinum | Gurugram | T1 | null | TRUE |
| HCCB10005 | Bronze | Indore | T1 | null | TRUE |

---

## Steps 2–4 → the source DataFrame

Watermark filter keeps only rows with `read_timestamp > T1`, dedupe keeps one row per
`customer_code`, then the three SCD2 columns are constructed.

**`silver_df` (3 rows):**

| customer_code | outlet_type | city | start_date | end_date | is_current |
|---|---|---|---|---|---|
| HCCB10002 | **Platinum** | Pune | T2 | null | TRUE |
| SLMG10003 | Iron | **Jodhpur** | T2 | null | TRUE |
| HCCB10006 | Gold | Ahmedabad | T2 | null | TRUE |

Every row here is shaped as a **new open version**: `start_date = T2`, `end_date` open,
`is_current = true`. Whether it ends up closing an old row or standing alone is decided by
the merges, not by the DataFrame.

---

## Step 5 — close the changed rows

```
ON  target.customer_code = source.customer_code AND target.is_current = true
WHEN MATCHED AND (customer_name <> ... OR outlet_type <> ... OR city <> ...)
THEN UPDATE SET end_date = source.start_date, is_current = false
```

Two conditions do two different jobs:

- **`target.is_current = true`** in the join — only ever touch the row that is currently
  open, never a version already closed
- **the column comparison** in `whenMatchedUpdate` — only close rows whose *business data*
  actually differs, so a rerun with no changes does nothing

**Silver after Step 5 (still 5 rows):**

| customer_code | outlet_type | city | start_date | end_date | is_current | |
|---|---|---|---|---|---|---|
| HCCB10001 | Bronze | Lucknow | T1 | null | TRUE | untouched |
| 🔴 HCCB10002 | Gold | Pune | T1 | 🔴 **T2** | 🔴 **FALSE** | ⬅ CLOSED |
| 🔴 SLMG10003 | Iron | Jaipur | T1 | 🔴 **T2** | 🔴 **FALSE** | ⬅ CLOSED |
| SLMG10004 | Platinum | Gurugram | T1 | null | TRUE | untouched |
| HCCB10005 | Bronze | Indore | T1 | null | TRUE | untouched |

`end_date` is set to the **new version's `start_date`**, not to `current_timestamp()`.
That makes the periods join edge to edge with no gap — essential for point-in-time joins.

`HCCB10006` matched nothing, so Step 5 ignored it entirely.

⚠ **Right now `HCCB10002` and `SLMG10003` have no open row.** The dimension is temporarily
incomplete. Step 6 is what makes it whole again — which is why the two steps must both run.

---

## Step 6 — insert the new versions

```
ON  target.customer_code = source.customer_code AND target.is_current = true
WHEN NOT MATCHED THEN INSERT *
```

Step 5 just set `is_current = false` on the changed rows, so they no longer match this join
→ they fall to NOT MATCHED → inserted as new open versions.
`HCCB10006` never existed → also NOT MATCHED → inserted.
The three unchanged outlets still have open rows → they MATCH → nothing happens to them.

**Silver after Step 6 — final, 8 rows:**

| customer_code | outlet_type | city | start_date | end_date | is_current | |
|---|---|---|---|---|---|---|
| HCCB10001 | Bronze | Lucknow | T1 | null | TRUE | |
| HCCB10002 | Gold | Pune | T1 | T2 | FALSE | history |
| 🟢 **HCCB10002** | 🟢 **Platinum** | Pune | 🟢 **T2** | null | 🟢 **TRUE** | ⬅ NEW VERSION |
| SLMG10003 | Iron | Jaipur | T1 | T2 | FALSE | history |
| 🟢 **SLMG10003** | Iron | 🟢 **Jodhpur** | 🟢 **T2** | null | 🟢 **TRUE** | ⬅ NEW VERSION |
| SLMG10004 | Platinum | Gurugram | T1 | null | TRUE | |
| HCCB10005 | Bronze | Indore | T1 | null | TRUE | |
| 🟢 **HCCB10006** | Gold | Ahmedabad | 🟢 **T2** | null | 🟢 **TRUE** | ⬅ NEW OUTLET |

**Legend:** 🟡 incoming change · 🔴 closed by Step 5 · 🟢 inserted by Step 6

---

## What each row now tells you

`HCCB10002` was **Gold from T1 until T2, and Platinum from T2 onward.** That fact exists
nowhere in an SCD1 table — the overwrite would have erased Gold completely.

`SLMG10003` sold from **Jaipur until T2, then Jodhpur.** Rerun last month's state-level sales
rollup and the volume stays in Rajasthan where it belongs, instead of being retroactively
re-credited to the new city.

---

## Validation

The invariant that must hold after every SCD2 run — **exactly one open row per key**:

```sql
SELECT customer_code, COUNT(*)
FROM   slowly_changing_dim.silver.scd2
WHERE  is_current = true
GROUP BY customer_code
HAVING COUNT(*) > 1;
```

Zero rows returned = correct.
Rows returned = Step 5 failed to close something, and the dimension now double-counts.

Point-in-time join, the reason SCD2 exists:

```sql
SELECT f.*, d.outlet_type, d.city
FROM   fact_sales f
JOIN   silver.scd2 d
  ON   f.customer_code = d.customer_code
 AND   f.sale_date >= d.start_date
 AND   (f.sale_date <  d.end_date OR d.end_date IS NULL);
```

Each sale is attributed to the outlet's attributes **as they were on the day of the sale**.

---

## Quick contrast with SCD1

| | SCD1 | SCD2 |
|---|---|---|
| Operations | 1 merge | 2 merges |
| Rows after this run | 6 | 8 |
| `HCCB10002` rows | 1 (Platinum) | 2 (Gold closed, Platinum open) |
| Old value | gone | preserved |
| Extra columns | none | `start_date`, `end_date`, `is_current` |
| Key identifies a row? | yes | **no** — needs a surrogate key |

Steps 1–4 are identical for both. Only steps 5–6 differ.

---

## Two things to tighten

**Step 6's join depends on Step 5 having run.** Keying on `(customer_code, start_date)`
instead of `is_current` makes it ask the real question — "does this exact version already
exist?" — and makes the step rerun-safe on its own.

**`end_date = NULL` for the open row.** A sentinel such as `9999-12-31` lets every
point-in-time join be a clean `BETWEEN` instead of needing `OR end_date IS NULL` forever.

**Still to add:** a surrogate key. Once a `customer_code` has several rows, it can no longer
identify one — `xxhash64(customer_code || start_date)` is the standard fix, and it is
deterministic so reruns produce identical keys.

In [0]:
# %sql
# DROP TABLE IF EXISTS slowly_changing_dim.silver.scd2;

In [0]:
# ── Parameter: Watermark Timestamp ──────────────────────────────
# Leave EMPTY for full load. Provide a timestamp (e.g. "2026-09-19 00:00:00")
# to process only rows with read_timestamp > the given value.
# dbutils.widgets.text("watermark_timestamp", "", "Watermark Timestamp (empty = full load)")

watermark_ts = dbutils.widgets.get("watermark_timestamp")
is_incremental = bool(watermark_ts and watermark_ts.strip())

print(f"Watermark timestamp : '{watermark_ts}'")
print(f"Load mode          : {'INCREMENTAL (upsert)' if is_incremental else 'FULL LOAD (overwrite)'}")

In [0]:
## 1. load the Source Table
bronze_df = spark.table("slowly_changing_dim.bronze.scd2")

display(bronze_df)

In [0]:
## 1. load the Source Table
bronze_df = spark.table("slowly_changing_dim.bronze.scd2")

## 2. first filter the records whether backfill or incremental
if is_incremental:
    bronze_df = (
        bronze_df.filter(
            F.col("read_timestamp") > F.lit(watermark_ts).cast('timestamp')
        )
    )

w = Window.partitionBy('customer_code').orderBy(F.col('read_timestamp').desc())
bronze_df = (
    bronze_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn')
)

## Create the silver table with SCD Type 2 Columns
silver_df = (
    bronze_df
    .withColumn('start_date', F.col('read_timestamp'))
    .withColumn('end_date', F.lit(None).cast('timestamp'))
    .withColumn('is_current', F.lit(True))
    .drop('read_timestamp')
)

display(silver_df)

In [0]:
## 1. load the Source Table
silver_table = spark.table("slowly_changing_dim.silver.scd2")

display(silver_table)

In [0]:
## 1. load the Source Table
bronze_df = spark.table("slowly_changing_dim.bronze.scd2")

## 2. first filter the records whether backfill or incremental
if is_incremental:
    bronze_df = (
        bronze_df.filter(
            F.col("read_timestamp") > F.lit(watermark_ts).cast('timestamp')
        )
    )

## 3. Deduplicate the records, keep the latest records only
w = Window.partitionBy('customer_code').orderBy(F.col('read_timestamp').desc())
bronze_df = (
    bronze_df
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn')
)

## Create the silver table with SCD Type 2 Columns
silver_df = (
    bronze_df
    .withColumn('start_date', F.col('read_timestamp'))
    .withColumn('end_date', F.lit(None).cast('timestamp'))
    .withColumn('is_current', F.lit(True))
    .drop('read_timestamp')
)

## Destination Table
table_exists = spark.catalog.tableExists('slowly_changing_dim.silver.scd2')

if is_incremental and table_exists:
    print("upsert old records and append new records")
    
    silver_table = DeltaTable.forName(spark, 'slowly_changing_dim.silver.scd2')

    ## First matched, update the end_date of the old records to current date
    (
        silver_table.alias('target')
        .merge(
            silver_df.alias('source'),
            F.expr("""
                target.customer_code = source.customer_code AND
                target.is_current == true       
            """)
        ).whenMatchedUpdate(
            condition= """
                target.customer_name != source.customer_name OR
                target.outlet_type != source.outlet_type OR
                target.outlet_segment != source.outlet_segment OR
                target.city != source.city OR
                target.state != source.state OR
                target.is_current != source.is_current
            """,           
            set = {
                "target.end_date": "source.start_date",
                "is_current": "false"
            }
        ).execute()
    )    

    ## First matched, update the end_date of the old records to current date
    (
        silver_table.alias('target')
        .merge(
            silver_df.alias('source'),
            condition = F.expr("""
                target.customer_code = source.customer_code AND
                target.is_current = true       
            """)
        ).whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        silver_df
        .write
        .mode('overwrite')
        .option('overwriteSchema', "true")
        .saveAsTable('slowly_changing_dim.silver.scd2')
    )

In [0]:
display(spark.table('slowly_changing_dim.silver.scd2').limit(10))